# Sisyphus Humanoid — Evaluation & Export Notebook

Load trained checkpoints, run evaluation episodes, render preview MP4s,
and export Blender-friendly data for cinematic rendering.

## 1. Setup

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 imageio[ffmpeg]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

REPO_URL = 'https://github.com/Mrepp/Sisyphus.git'
REPO_DIR = '/content/Sisyphus'
DRIVE_DIR = '/content/drive/MyDrive/sisyphus'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, REPO_DIR)

## 2. Load Checkpoint

In [ ]:
import glob

# List available checkpoints
checkpoints = sorted(glob.glob(f'{DRIVE_DIR}/checkpoints/*.zip'))
print(f'Found {len(checkpoints)} checkpoints:')
for cp in checkpoints:
    print(f'  {os.path.basename(cp)}')

In [ ]:
from stable_baselines3 import PPO
from env.sisyphus_env import SisyphusEnv

# Select checkpoint (use final or specific step)
CHECKPOINT_PATH = f'{DRIVE_DIR}/checkpoints/sisyphus_ppo_final'
# Or: CHECKPOINT_PATH = checkpoints[-1].replace('.zip', '')  # latest

model = PPO.load(CHECKPOINT_PATH)
print(f'Loaded: {CHECKPOINT_PATH}')

## 3. Evaluate & Render Preview

In [ ]:
from IPython.display import Video, display
from render.preview_renderer import PreviewRenderer
from logging_utils.trajectory_logger import TrajectoryLogger
import numpy as np

def evaluate_checkpoint(model, slope_deg, rock_mass, max_steps=1000, label=''):
    """Run one eval episode, render preview, save trajectory."""
    env = SisyphusEnv(slope_deg=slope_deg, rock_mass=rock_mass, max_steps=max_steps)
    
    # Trajectory logging
    traj_logger = TrajectoryLogger(save_dir='/content/eval_replays')
    obs, info = env.reset()
    
    total_reward = 0
    for step in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        
        traj_logger.record_step(
            model=env.model, data=env.data, action=action,
            reward=reward, rock_body_id=env._rock_id,
            height_offset=info.get('total_height_accumulated', 0.0))
        
        if terminated or truncated:
            break
    
    # Save trajectory
    metadata = {'slope': slope_deg, 'mass': rock_mass, 'fps': 1.0/(env.model.opt.timestep*5)}
    traj_path = traj_logger.save_episode(0, checkpoint_id=0, metadata=metadata)
    
    # Render MP4
    renderer = PreviewRenderer(env.model, width=960, height=540)
    mp4_path = f'/content/eval_{label}.mp4'
    renderer.render_trajectory(traj_path, mp4_path, fps=30)
    renderer.close()
    env.close()
    
    print(f'{label}: reward={total_reward:.1f}, steps={step+1}, '
          f'rock_height={info["rock_height"]:.3f}m')
    return mp4_path, traj_path

In [ ]:
# Evaluate at different difficulty levels
configs = [
    (0.0,  5.0,  'flat_light'),
    (5.0,  8.0,  'slope5_medium'),
    (10.0, 12.0, 'slope10_heavy'),
    (15.0, 10.0, 'slope15_infinite'),
]

results = []
for slope, mass, label in configs:
    mp4, traj = evaluate_checkpoint(model, slope, mass, max_steps=500, label=label)
    results.append((mp4, traj, label))

In [ ]:
# Display preview videos
for mp4_path, _, label in results:
    print(f'\n--- {label} ---')
    display(Video(mp4_path, embed=True, width=960))

## 4. Export for Blender

In [ ]:
from export.blender_export import export_for_blender

BLENDER_DIR = f'{DRIVE_DIR}/blender_export'

for _, traj_path, label in results:
    out_dir = f'{BLENDER_DIR}/{label}'
    export_for_blender(traj_path, out_dir, target_fps=60.0)
    print(f'Exported: {out_dir}')
    for f in os.listdir(out_dir):
        size = os.path.getsize(os.path.join(out_dir, f))
        print(f'  {f} ({size/1024:.1f} KB)')

## 5. Film Structure: Checkpoint Progression

Load multiple checkpoints across training to show the learning arc:
- Early: falls repeatedly, rock crushes agent
- Mid: brief pushes, hesitation
- Final: rhythmic continuous motion

In [ ]:
# Select checkpoints for film narrative
film_checkpoints = [
    # (checkpoint_path, slope, mass, label)
    # Uncomment and fill with your actual checkpoint paths:
    # (f'{DRIVE_DIR}/checkpoints/sisyphus_ppo_500000_steps', 0.0, 5.0, 'early'),
    # (f'{DRIVE_DIR}/checkpoints/sisyphus_ppo_5000000_steps', 5.0, 8.0, 'learning'),
    # (f'{DRIVE_DIR}/checkpoints/sisyphus_ppo_15000000_steps', 10.0, 12.0, 'mid'),
    # (f'{DRIVE_DIR}/checkpoints/sisyphus_ppo_final', 15.0, 10.0, 'final'),
]

for cp_path, slope, mass, label in film_checkpoints:
    cp_model = PPO.load(cp_path)
    
    # Use infinite mode for final checkpoint
    infinite = (label == 'final')
    env = SisyphusEnv(slope_deg=slope, rock_mass=mass, max_steps=2000,
                      infinite_mode=infinite)
    
    traj_logger = TrajectoryLogger(save_dir=f'{BLENDER_DIR}/film')
    obs, info = env.reset()
    
    for step in range(2000):
        action, _ = cp_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        traj_logger.record_step(
            model=env.model, data=env.data, action=action,
            reward=reward, rock_body_id=env._rock_id,
            height_offset=info.get('total_height_accumulated', 0.0))
        if terminated or truncated:
            break
    
    metadata = {'slope': slope, 'mass': mass, 'label': label,
                'fps': 1.0/(env.model.opt.timestep*5)}
    traj_path = traj_logger.save_episode(0, checkpoint_id=0, metadata=metadata)
    
    # Export at 60fps for Blender
    export_for_blender(traj_path, f'{BLENDER_DIR}/film/{label}', target_fps=60.0)
    print(f'{label}: {step+1} steps, exported to {BLENDER_DIR}/film/{label}')
    env.close()

## 6. Download Exports

In [ ]:
# Create a zip of all Blender exports for download
import shutil

zip_path = shutil.make_archive('/content/blender_export', 'zip', BLENDER_DIR)
print(f'Created: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')

from google.colab import files
files.download(zip_path)

In [ ]:
# Summary of exported data
print('=== Blender Export Contents ===')
for root, dirs, fls in os.walk(BLENDER_DIR):
    level = root.replace(BLENDER_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in fls:
        size = os.path.getsize(os.path.join(root, f))
        print(f'{subindent}{f} ({size/1024:.1f} KB)')